<a href="https://colab.research.google.com/github/aleezasaleem/45assignment/blob/md/clothe_app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Clothe Agent By Abu Bakar* :

i created a *e-commerce customer support agent* for boys using Python LangGrpah  Gemini_Api_Key mockApi


In [1]:
%pip install --quiet -U langchain_core langgraph langchain_google_genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.2/138.2 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.5/41.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 4.2 MB/s eta 0:00:00


In [2]:
import os
from langchain_core.messages import SystemMessage, HumanMessage, RemoveMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import MessagesState, StateGraph, START, END
from langgraph.graph.state import CompiledStateGraph
from langgraph.checkpoint.memory import MemorySaver
import requests


GOOGLE_API_KEY = os.environ.get("GEMINI_API_KEY")

model = ChatGoogleGenerativeAI(model="gemini-1.5-flash")

# Define the MockAPI Base URL
MOCKAPI_URL = "https://675e967563b05ed0797a7ef4.mockapi.io/products"


In [4]:
# Extend MessagesState for custom chatbot state
from langgraph.graph import MessagesState

class ShopState(MessagesState):
    # Summary of the conversation
    summary: str = ""

    # User preferences (could include favorite products, size, etc.)
    preferences: dict = {}


In [5]:
# Tool-calling functions
def fetch_products():
    response = requests.get(MOCKAPI_URL)
    if response.status_code == 200:
        return response.json()
    return {"error": "Failed to fetch products"}

In [6]:
def fetch_product_by_id(product_id):
    response = requests.get(f"{MOCKAPI_URL}/{product_id}")
    if response.status_code == 200:
        return response.json()
    return {"error": f"Failed to fetch product with ID {product_id}"}

In [7]:
def create_product(name, price, size, stock):
    response = requests.post(MOCKAPI_URL, json={"name": name, "price": price, "size": size, "stock": stock})
    if response.status_code == 201:
        return response.json()
    return {"error": "Failed to create product"}

In [8]:
def call_model(state: ShopState):
    """Core conversational logic."""
    summary = state.get("summary", "")
    system_message = f"Conversation Summary:\n{summary}" if summary else "You're a helpful assistant for a boys' clothing store."
    messages = [SystemMessage(content=system_message)] + state["messages"]
    response = model.invoke(messages)
    return {"messages": response}

In [9]:
def summarize_conversation(state: ShopState):
    """Summarize the conversation."""
    summary = state.get("summary", "")
    prompt = (
        f"Summarize this conversation about boys' clothing shop:\n\n{summary}" if summary
        else "Summarize this conversation for a boys' clothing shop assistant."
    )
    messages = state["messages"] + [HumanMessage(content=prompt)]
    response = model.invoke(messages)
    # Retain only the last 2 messages in full detail
    delete_messages = [RemoveMessage(id=m.id) for m in state["messages"][:-2]]
    return {"summary": response.content, "messages": delete_messages}

In [10]:
def should_continue(state: ShopState):
    """Determine if we should summarize or continue the conversation."""
    if len(state["messages"]) > 6:
        return "summarize_conversation"
    return END

In [11]:
workflow = StateGraph(ShopState)

# Add nodes
workflow.add_node("conversation", call_model)
workflow.add_node("summarize_conversation", summarize_conversation)

# Define workflow edges
workflow.add_edge(START, "conversation")
workflow.add_conditional_edges("conversation", should_continue)
workflow.add_edge("summarize_conversation", END)

# Compile the graph with memory
memory = MemorySaver()
graph = workflow.compile(checkpointer=memory)

In [12]:


# Chatbot runtime logic
def run_chatbot():
    """Run the chatbot."""
    config = {"configurable": {"thread_id": "customer-session"}}

    print("Welcome to the Boys' Clothing Shop Chatbot! Type 'exit' to end the conversation.")

    while True:
        user_input = input("You: ")
        if user_input.lower() == "exit":
            print("Goodbye! Thank you for visiting.")
            break

        # Create a HumanMessage and invoke the graph
        input_message = HumanMessage(content=user_input)
        output = graph.invoke({"messages": [input_message]}, config)

        # Get the chatbot's response
        bot_response = output["messages"][-1].content
        print(f"Chatbot: {bot_response}")

        # Show updated summary
        state = graph.get_state(config)
        if "summary" in state.values:
            print("\nCurrent Summary:")
            print(state.values["summary"])




In [ ]:
# Run the chatbot
run_chatbot()